# **CSE 291 / DSC 291: Machine Learning Systems — Programming Assignment 1**

Welcome to PA1! In this assignment, you will build a **mini deep learning framework from scratch** and use it to train a **transformer language model** that generates text.

### What You Will Learn

- How **automatic differentiation** works under the hood — the same technique that powers PyTorch, TensorFlow, and JAX
- How to implement forward and backward passes for core operations: matrix multiply, softmax, layer normalization, and more
- How a **decoder-only transformer** (the architecture behind GPT, LLaMA, etc.) is constructed from these primitives
- How **next-token prediction**, **cross-entropy loss**, and **greedy decoding** work in a language model

### What You Will Build

1. **An autodiff engine** (`auto_diff.py`): You will implement operators (matmul, softmax, LayerNorm, etc.), a graph evaluator, and reverse-mode gradient computation.
2. **A transformer language model** (`transformer.py`): Using only your autodiff engine (no `loss.backward()`!), you will build causal self-attention, a decoder layer, a training loop, and a text generation function.
3. By the end, you will **train your model to memorize 10 sentences** and run generation demos directly from this notebook.

### Logistics

- **Language:** Python only — no C++ or CUDA required.
- **Hardware:** No GPU needed. Everything runs on CPU in under a minute.
- **Collaboration:** This assignment must be completed individually. Do not post your work on public platforms.
- **Submission:** Submit the entire PA1 folder as a zip to Gradescope. Refer to the final section for details.

### Testing and Grading

We have provided public test scripts in `tests/` that you can run to verify your implementation. Your grade will also depend on private tests. You may submit multiple times — only the last submission before the deadline counts. We strongly encourage writing your own tests as well.

## **Overview**

Automatic differentiation forms the core technique for training machine learning models. In this assignment, you are required to develop a basic automatic differentiation system from scratch. Several functions will be given to you, and you will focus on creating standard operations used in transformers and other ML architectures - namely LayerNorm, ReLU, Softmax, among others.

Let's first go through a run-down of how autodiff works.

The automatic differentiation algorithm in this assignment operates using a computational graph. A computational graph visually represents the sequence of operations needed to compute an expression. You can reference lecture 2, slide 37 for a quick example on how this graph works.

Let's begin by exploring the fundamental concepts and data structures used in the framework. A computational graph is composed of nodes, each representing a distinct computation step in the evaluation of the entire expression.

Each node consists of three components, as shown in auto_diff.py line 6:

*   an operation (field op), specifying the type of computation the node performs.
*   a list of input nodes (field inputs), detailing the sources of input for the computation.
*   optionally, additional "attributes" (field attrs), which vary depending on the node's operation.

These attributes will be discussed in more detail later in this section.

Input nodes in a computational graph can be defined using ad.Variable. For instance, the input variable nodes $x_1$ and $x_2$ might be set up as follows:

```python
import auto_diff as ad

x1 = ad.Variable(name="x1")
x2 = ad.Variable(name="x2")
```

In auto_diff.py (line 81), the ad.Variable class is used to create a node with the operation placeholder and a specified name. Input nodes have empty inputs and attrs:

```python
class Variable(Node):
    def __init__(self, name: str) -> None:
        super().__init__(inputs=[], op=placeholder, name=name)
```

Here, the placeholder operation signifies that the input variable node does not perform any computation. Apart from placeholder, there are other operations defined in auto_diff.py, like add and matmul. You should not create your own instances of these ops.

Consider the example where
$y = x_1 * x_2 + x_1$. With x1 and x2 already established as input variables, the rest of the graph can be defined using just one line of Python:

```python
y = x1 * x2 + x1
```

This code first creates a node with the operation mul, taking x1 and x2 as its inputs. It then constructs another node with add, which utilizes the result of the multiplication node along with x1 as inputs. Consequently, this computational graph ultimately comprises four nodes.

**Important Note**

It's important to note that a computational graph (e.g., the four nodes we defined) does not inherently store the actual values of its nodes. The structure of this assignment aligns with the TensorFlow v1 approach that was covered in our lectures. This method contrasts with frameworks like PyTorch, where input tensor values are specified upfront, and the values of intermediate tensors are computed immediately as they are defined. 

In our framework, values are instead provided at evaluation time — you first define the graph structure, then supply input values through the `Evaluator.run` method (described below) to compute results.


#### **Evaluator**

Here's a walkthrough of how Evaluator works. The constructor of Evaluator accepts a list of nodes that it needs to evaluate. By initiating an Evaluator with:

```evaluator = ad.Evaluator(eval_nodes=[y])```

you are essentially setting up an Evaluator instance designed to compute the value of y. To calculate this, input tensor values are provided via the `Evaluator.run` method, which you will implement. **These input tensors are assumed to be of type `torch.Tensor` throughout this assignment.** Here's how it works:

Coming back to the example $y = x_1 * x_2 + x_1$ , to calculate the value of the output y given the inputs x1 and x2, we do: 
```python
import torch

x1_value = torch.tensor(2.0)
x2_value = torch.tensor(3.0)
y_value = evaluator.run(input_values={x1: x1_value, x2: x2_value})

```

In this process, the run method takes the input values using a dictionary of the form `Dict[Node, torch.Tensor]`, calculates the value of the node y internally, and outputs the result. For instance, with the input values 2 * 3 + 2 = 8, the expected result for y_value would be `torch.tensor(8.0)`.

The `Evaluator.run` method is responsible for the forward computation of nodes. Building on what was discussed in the lectures, to calculate the gradient of the output with respect to each input node within a computational graph, we enhance the forward graph with an additional backward component. By integrating both forward and backward graphs, and providing values for the input nodes, the Evaluator can compute the output value, the loss value, and the gradient values for each input node in a single execution of `Evaluator.run`.

You are tasked with implementing the function ```gradients(output_node: Node, nodes: List[Node]) -> List[Node]``` for some of the operators found in auto_diff.py. This function constructs the backward graph needed for gradient computation. It accepts an output node—typically the node representing the loss function in machine learning applications, where the gradient is preset to 1. It also takes a list of nodes for which gradients are to be computed and returns a list of gradient nodes corresponding to each node in the input list.

Returning to our earlier example, once you have implemented the gradients function, you can use it to calculate the gradients of $y$ with respect to $x_1$ and $x_2$. This is done by running:

```x1_grad, x2_grad = ad.gradients(output_node=y, node=[x1, x2])```

to obtain the respective gradients. Following this, you can set up an Evaluator with nodes y, x1_grad, and x2_grad. This allows you to use the Evaluator.run method to compute both the output value and the gradients for the input nodes.

Before you start working on the assignment, let's clarify how operations (ops) work. Within auto_diff.py, each op is equipped with three methods:

```__call__(self, **kwargs) -> Node```:

*  accepts input nodes (and attributes), creates a new node utilizing this op, and returns the newly created node.

```compute(self, node: Node, input_values: List[torch.Tensor])-> torch.Tensor```

*  processes the specified node along with its input values and delivers the resultant node value.

```gradient(self, node: Node, output_grad: Node) -> List[Node]```

*  receives a node and its gradient node, returning the partial adjoint nodes for each input node.

In short, each `Op` handles a single node: `compute` evaluates the forward pass and calculates the actual output value, while `gradient` builds the backward path for that node. `Evaluator.run` and the `gradients` function orchestrate these across the entire graph. Accordingly, your `Evaluator.run` implementation should effectively utilize each op's `compute`, and your `gradients` implementation should call each op's `gradient`.

## **Question 1: Auto Diff Library (45 pt)**

### Part 1: Operators (32 pt)
In this problem, you will finish several operators in the autodiff.py class. Your goal is to implement the compute (forward) and gradient (backwards) function for these operators, with a few examples provided to you, such as `AddOp`, `AddByConstOp`, `MulOp` and `MulByConstOp`, and more. We have also implemented several other functions, such as `BroadcastOp` which will be useful in Question 3, but you can ignore them for now.

The list of operators that you will need to implement are:

*   `DivOp`
*   `DivByConstOp`
*   `TransposeOp`
*   `ReLUOp`
*   `SqrtOp`
*   `PowerOp`
*   `MeanOp`
*   `MatMulOp`
*   `SoftmaxOp`
*   `LayerNormOp`

You will find that most of your time will be spent on Matmul, SoftMax, and the LayerNorm operators. In turn, these 3 operators alone will be half of the points for this section.

### Part 2: Evaluator (13 pt)
You will also complete the entire `Evaluator` class. The details of the `Evaluator` class and what we want to achieve are given in the introduction section. We have provided a framework, including a topological sort method that you may choose to implement to help you more easily complete this class.

There are several tests provided to ensure your operators are working.

To run our sample testing library, you can use the commands:

```pytest tests/test_auto_diff_node_forward.py```

```pytest tests/test_auto_diff_node_backward.py```

```pytest tests/test_auto_diff_graph_forward.py```

```pytest tests/test_auto_diff_graph_backward.py```

Feel free to also edit these test files to include your own test cases!

**NOTE: These tests are not fully comprehensive, so we highly encourage you to write your own tests and ensure that your implemented operators are working.** You may find yourself going back and forth between this part and question 2 to ensure your operators are fully correct.


In [1]:
!pytest tests/test_auto_diff_node_forward.py
!pytest tests/test_auto_diff_node_backward.py
!pytest tests/test_auto_diff_graph_forward.py
!pytest tests/test_auto_diff_graph_backward.py

============================= test session starts ==============================
platform darwin -- Python 3.10.20, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/ziangli/Private/Courses/CSE291A-ML-Sys/ML-sys-pas/pa1
collected 13 items                                                             

tests/test_auto_diff_node_forward.py .............                       [100%]

============================== 13 passed in 4.57s ==============================
============================= test session starts ==============================
platform darwin -- Python 3.10.20, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/ziangli/Private/Courses/CSE291A-ML-Sys/ML-sys-pas/pa1
collected 12 items                                                             

tests/test_auto_diff_node_backward.py ............                       [100%]

============================== 12 passed in 0.44s ==============================
============================= test session starts ==============================
platfor

## **Question 2: Transformer Language Model & Training (45 pt)**

In this assignment, you will implement core components of a **decoder-only Transformer language model** using our custom automatic differentiation framework. You will train it to **overfit** a small set of 10 sentences, then use it to **generate text** autoregressively.

This exercise mirrors the core training loop of modern large language models (LLMs) — just at a much smaller scale so everything runs comfortably on a CPU.

### Provided Operations
Your implementation should only use operations **THAT YOU IMPLEMENTED** from the **auto_diff** module, not pre-built torch methods. Any code that uses PyTorch's built-in autograd (e.g., `loss.backward()`) will receive zero credit.

### Background

A **decoder-only transformer** processes a sequence of tokens and predicts the next token at each position. The key difference from an encoder is **causal masking**: each position can only attend to itself and earlier positions, preventing information from "leaking" from the future.

**Word-level tokenizer.** We provide a simple word-level tokenizer that splits sentences on whitespace and maps each unique word to an integer index. This is conceptually the same idea as real LLM tokenizers (BPE, SentencePiece, etc.), but much simpler — our "vocabulary" is just whole words. A special `<pad>` token (index 0) exists for use during generation but never appears in the training data.

**Training setup.** We provide 10 short sentences, each exactly 5 words long. The model is small enough (~39K parameters) to train on CPU in under a minute. Since we have very little data, the goal is to **overfit** — the model should memorize these 10 sentences and be able to reproduce them from a 2-word prompt.

### Part 1: Causal Self-Attention (10 pt)

Implement single-head **causal** self-attention in the `causal_self_attention()` function:

$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

$$\text{scores} = \frac{Q K^T}{\sqrt{d_{\text{model}}}} + \text{mask}$$

$$\text{output} = \text{Softmax}(\text{scores}) \cdot V \cdot W_O$$

The **causal mask** is a matrix with 0 for allowed positions and $-10^9$ for future positions, which drives those attention weights to ~0 after softmax. The mask is pre-tiled to `(batch, seq_len, seq_len)` and passed as an input variable.

### Part 2: Decoder Layer (5 pt)

Implement `decoder_layer()`, which combines self-attention with a feed-forward network, using **residual connections** and **layer normalization**:

1. `attn_out = causal_self_attention(X, ...)`
2. `h = LayerNorm(X + attn_out)` ← residual + norm
3. `ff_out = ReLU(h @ W_ff1) @ W_ff2` ← feed-forward
4. `output = LayerNorm(h + ff_out)` ← residual + norm

### Part 3: LM Forward Pass + Loss (10 pt)

**`transformer_lm()` (5 pt):** Build the full forward pass:

1. Token embedding: `X_onehot @ W_embed` (one-hot word index × embedding matrix)
2. Add position embeddings (pre-tiled to batch size, passed as input)
3. One decoder layer
4. Output projection: `h @ W_head` → logits of shape `(batch, seq_len, vocab_size)`

**`cross_entropy_loss()` (5 pt):** Compute the average cross-entropy loss for next-token prediction:

$$\text{loss} = -\frac{1}{N} \sum \text{targets} \cdot \log(\text{softmax}(\text{logits}))$$

where $N$ is the total number of target tokens (`BATCH_SIZE × SEQ_LEN`).

### Part 4: Training & Generation (20 pt)

**`sgd_epoch()` (10 pt):** Data preparation and the forward/backward pass are provided. You only need to implement the **weight update loop**:
- Each gradient has a leading batch dimension — sum over dim 0 before updating
- `new_W = W - lr * grad.sum(dim=0)`

**`generate()` (10 pt):** Graph setup and an inference helper `run_forward()` are provided. You only need to implement the **generation loop**:
- Encode the prompt
- Loop: call `run_forward(tokens)`, argmax the logits at the last valid position, append the new token
- Return the decoded text

**`train_model()`** is **fully provided** — it wires together the forward graph, backward graph, evaluator, weight initialization, and training loop. You do not need to modify it. Just make sure the functions it calls (`transformer_lm`, `cross_entropy_loss`, `sgd_epoch`, `generate`) are correct.

**Recommended order:** Read `train_model()` first to understand the pipeline. Then implement bottom-up: attention → decoder layer → forward pass → loss → weight update → generation loop.

### Submission, Testing, and Expected Results

Write your implementation in `transformer.py`. In addition to code, save your trained model weights as `weights.pt` using `save_weights(weights)`.

During evaluation, we will load your submitted `transformer.py` and `weights.pt`, then run `tests/test_transformer.py`. The public test checks that your weights can be loaded, that the tensor shapes are correct, and that generation from the first two words of each training sentence reproduces at least **8 out of 10** training sentences.

You can run training from the notebook or from the command line. Both should save a `weights.pt` file.

Command-line usage:
- `python transformer.py` trains the model and saves `weights.pt`
- `python transformer.py --playground` trains the model, saves `weights.pt`, and then launches the interactive playground

Notebook usage:
- The next code cell runs training and saves `weights.pt`
- The following code cell runs `tests/test_transformer.py`

Expected behavior:
- Training loss should decrease over 500 epochs, reaching near 0
- Generation from 2-word prompts should reproduce at least **8 out of 10** training sentences
- `python -m pytest tests/test_transformer.py` should pass after `weights.pt` has been saved

Scoring for Q2:
- Part 1: 10 pt
- Part 2: 5 pt
- Part 3: 10 pt
- Part 4: 20 pt
- Q2 required total: 45 pt
- Bonus section below: 10 pt

Together with Q1 (45 pt), PA1 is graded out of **100 total points**.

No GPU is needed. Training should take **under 2 minutes** on a modern CPU.

### Key Differences from a Standard Transformer

| Simplification | Why |
|---------------|-----|
| Single attention head | Keeps implementation simple |
| No bias terms | Fewer parameters to manage |
| Word-level tokenizer | Simple vocabulary, no subword splitting |
| Full forward pass for generation | No KV cache to implement |
| Pre-tiled position embeddings | Avoids complex broadcasting inside the graph |


### Q2 Notebook Workflow

The cells below give a complete notebook workflow for Q2:

1. Train the model and save `weights.pt`
2. Run the public transformer test
3. Complete the bonus writeup on decoding complexity without KV cache
4. Load the saved weights and try a few generation prompts


In [2]:
import transformer

weights = transformer.train_model()
transformer.save_weights(weights, "weights.pt")
print("Saved weights to weights.pt")


Epoch  20/500:  loss = 1.6608
Epoch  40/500:  loss = 0.6718
Epoch  60/500:  loss = 0.3526
Epoch  80/500:  loss = 0.2285
Epoch 100/500:  loss = 0.1660
Epoch 120/500:  loss = 0.1293
Epoch 140/500:  loss = 0.1054
Epoch 160/500:  loss = 0.0889
Epoch 180/500:  loss = 0.0768
Epoch 200/500:  loss = 0.0675
Epoch 220/500:  loss = 0.0602
Epoch 240/500:  loss = 0.0543
Epoch 260/500:  loss = 0.0495
Epoch 280/500:  loss = 0.0455
Epoch 300/500:  loss = 0.0420
Epoch 320/500:  loss = 0.0391
Epoch 340/500:  loss = 0.0365
Epoch 360/500:  loss = 0.0343
Epoch 380/500:  loss = 0.0323
Epoch 400/500:  loss = 0.0305
Epoch 420/500:  loss = 0.0289
Epoch 440/500:  loss = 0.0275
Epoch 460/500:  loss = 0.0262
Epoch 480/500:  loss = 0.0250
Epoch 500/500:  loss = 0.0240

--- Generation test ---
  [OK] prompt='attention is' -> 'attention is all you need'
  [OK] prompt='the model' -> 'the model learns very fast'
  [OK] prompt='we will' -> 'we will overfit this data'
  [OK] prompt='gradients flow' -> 'gradients flow th

In [3]:
!pytest tests/test_transformer.py


============================= test session starts ==============================
platform darwin -- Python 3.10.20, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/ziangli/Private/Courses/CSE291A-ML-Sys/ML-sys-pas/pa1
collected 5 items                                                              

tests/test_transformer.py .....                                          [100%]

============================== 5 passed in 0.45s ===============================


### Bonus: Decoding Complexity and Repeated Computation (10 pt)

This bonus asks you to analyze the mathematical complexity of the current autoregressive decoding implementation and think about how repeated computation could be reduced. In our current `generate()` implementation, the model runs a **full forward pass over the entire sequence** every time a new token is generated.

For asymptotic analysis, treat sequence length as a variable, even though this assignment uses a small fixed sequence length in practice.

For this bonus, write a short response in `bonus.txt` that answers the following in mathematical terms:

1. What is the **time complexity per decoding step** of the current implementation without KV cache?
2. What is the **total time complexity** of generating $T$ new tokens with the current implementation without KV cache?
3. What is the **space complexity** of the current implementation without KV cache during generation?
4. Is there a way to reduce the repeated computation during decoding? Briefly describe the idea and how the time and space complexity would change. *(Hint: think about caching computations that are reused across decoding steps.)*

Requirements:
- No code is required for this bonus.
- Use Big-O notation and clearly define any symbols you use, such as $t$ for current context length, $T$ for the number of generated tokens, and $d$ for model dimension.
- Keep your answer concise but complete.
- We will grade this based on clarity and correctness of the mathematical reasoning.


## Submission

Congratulations, you have finished PA1!

Submit the entire `PA1` folder as a zipped file to Gradescope under the assignment **PA1**.

For this assignment, make sure your submission includes at least:
- `auto_diff.py`
- `transformer.py`
- `weights.pt`
- `bonus.txt` if you attempt the 10-point bonus

Q1 will be evaluated from your submitted `auto_diff.py`. Q2 will load your submitted `transformer.py` and `weights.pt`, then run `tests/test_transformer.py`. If `weights.pt` is missing, Q2 cannot be evaluated.

**Deadline: Wednesday, April 22 at 11:59 PM.**

Please note that you have a total of **5 free late days** across all assignments. You can allocate these late days to any assignments you choose, but submissions more than 5 days late will not receive credit without prior approval.

If you have any feedback on the difficulty of this assignment, feel free to leave it in `feedback.txt`. We appreciate your input!
